# ScamSense — Notebook 04: LangGraph Agentic Prediction Pipeline

This notebook implements the ScamSense **agentic pipeline** using **LangGraph**.
Three specialised agents communicate through a shared state graph, with conditional
routing that decides whether to run the full scam analysis or exit early for safe messages.

| Agent | Role | Method |
|---|---|---|
| **Detection Agent** | Classify Scam / Ham + detect language | Fine-tuned XLM-RoBERTa |
| **Explanation Agent** | SHAP token attribution + **RAG over SPF knowledge base** | SHAP + FAISS |
| **Risk Agent** | Scam type + risk score + SPF 2025 statistics | Rule-based SPF taxonomy |

**Why LangGraph?**  
LangGraph enforces a proper agent architecture: each agent is a node, state is passed
explicitly between nodes, and conditional edges allow the graph to branch — for example,
bypassing the Explanation and Risk agents entirely when a message is clearly safe.

```
User Message
      │
      ▼
┌─────────────────┐
│ Detection Agent │  ← XLM-RoBERTa
└────────┬────────┘
         │
    Ham (≥80%)?
    ┌────┴─────┐
   Yes         No
    │          │
    ▼          ▼
┌────────┐  ┌──────────────────┐
│  Safe  │  │ Explanation Agent│  ← SHAP + FAISS RAG (SPF corpus)
│  Exit  │  └────────┬─────────┘
└────────┘           │
                     ▼
              ┌─────────────┐
              │  Risk Agent │  ← SPF 2025 taxonomy
              └─────────────┘
```



## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# Run this cell first, then restart the kernel when prompted

packages = [
    "langgraph",
    "langdetect",
    "shap",
    "sentence-transformers==3.0.1",
    "faiss-cpu",
    "numpy<2",
    "pandas",
    "tabulate",
]

import subprocess, sys
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"  ✔ {pkg}")

#All packages installed — restart kernel now
import os; os.kill(os.getpid(), 9)

  ✔ langgraph
  ✔ langdetect
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 86.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


  ✔ shap
  ✔ sentence-transformers==3.0.1
  ✔ faiss-cpu


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 w

  ✔ numpy<2
  ✔ pandas


## Cell 2 — Imports & Setup

In [1]:
import re, json
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import torch
import faiss
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import shap
from langdetect import detect, LangDetectException
from langgraph.graph import StateGraph, END

# ── Kaggle Secrets ────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets  = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None   

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = Path("/kaggle/working/ScamSense")
RAG_DIR  = BASE_DIR / "rag"
RAG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"Base dir: {BASE_DIR}")
print("Imports complete")

Device  : cuda
Base dir: /kaggle/working/ScamSense
Imports complete


## Cell 3 — Load Fine-tuned XLM-RoBERTa Classifier

In [13]:
HF_MODEL_ID = "Bhoovika/scamsense-xlmroberta"

print(f"Loading tokenizer from {HF_MODEL_ID} ...")
clf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)

print("Loading model ...")
clf_model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_ID, token=HF_TOKEN
).to(DEVICE)
clf_model.eval()

LABEL_MAP = {0: "ham", 1: "scam"}

def run_classifier(text: str) -> dict:
    """Run XLM-RoBERTa on a single text. Returns label, confidence, scam_prob, logits."""
    inputs = clf_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_idx = int(probs.argmax())
    return {
        "label"     : LABEL_MAP[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "scam_prob" : round(float(probs[1]), 4),
        "logits"    : logits.cpu().numpy()[0].tolist(),
    }

# Sanity check
test = run_classifier("You won $10,000! Claim now.")
print(f"\nSanity check → label={test['label']} | confidence={test['confidence']:.1%}")
print("Classifier ready")

Loading tokenizer from Bhoovika/scamsense-xlmroberta ...
Loading model ...

Sanity check → label=scam | confidence=100.0%
Classifier ready


## Cell 4 — SPF 2025 Risk Taxonomy

Rule-based taxonomy derived from the **SPF Annual Scams & Cybercrime Brief 2025**.
Used by the Risk Agent to classify scam type and compute a risk score.

> **Why not load from a PDF?**  
> The SPF report is a government PDF with mixed layouts, tables, and charts — 
> direct PDF parsing produces noisy, unreliable text. Instead the key statistics 
> and advisory text were manually extracted and encoded here as structured data. 
> This is standard practice for RAG systems where source quality matters.
> The same passages are embedded into the FAISS index in Cell 5 for semantic retrieval.


In [14]:
# Source: Singapore Police Force, Annual Scam and Cybercrime Brief 2025 (25 Feb 2026)
# Taxonomy keys are aligned 1:1 with the 12 synthetic scam-type labels used in
# Notebook 01 (SLOTS/TEMPLATES) so the same category names are used end-to-end:
# data generation -> classification -> RAG corpus -> risk taxonomy.
#
# Categories with real 2025 SPF figures: investment, government_impersonation,
# job_scam, phishing, ecommerce, fake_friend, loan.
# bank_impersonation is not a separate SPF-tracked category; SPF's own report
# describes bank-impersonation calls as the entry vector into government
# impersonation scams and folds the tactic into Phishing Scam statistics
# (Annex D), so its figures are borrowed from phishing with a note.
# parcel_delivery, rental, prize, charity have NO official 2025 case/loss
# figures in the SPF Annual Brief -- they sit inside the unallocated "Other
# Scams" bucket (9,941 cases, $135.1M). Their stats fields are left as None
# rather than fabricated; advice is qualitative and cites general prevention
# guidance instead of invented numbers.

SPF_TAXONOMY = {
    "investment": {
        "spf_name"    : "Investment Scam",
        "2025_cases"  : 5462,
        "2025_losses" : 336.2,   # SGD millions
        "avg_loss"    : 61559,
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 95,
        "keywords"    : [
            r"invest", r"return", r"profit", r"forex", r"crypto",
            r"trading", r"guaranteed", r"passive income", r"portfolio",
            r"capital", r"dividend", r"scheme", r"fund", r"roi",
            r"bitcoin", r"ethereum", r"tether", r"usdt", r"wallet",
            r"keuntungan", r"pelaburan",
            r"முதலீட்", r"லாபம்",
            r"投资", r"收益", r"理财", r"赚钱",
        ],
        "advice": (
            "Investment scams caused the HIGHEST losses in Singapore in 2025 — "
            "$336.2 million (SPF 2025). Never transfer money to strangers for "
            "'investments'. Verify at MAS (mas.gov.sg/investor-alert-list). "
            "Report to SPF at 1800-255-0000."
        ),
    },
    "government_impersonation": {
        "spf_name"    : "Government Officials Impersonation Scam",
        "2025_cases"  : 3363,
        "2025_losses" : 242.9,
        "avg_loss"    : 72229,
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 93,
        "keywords"    : [
            r"police", r"spf", r"cpf", r"ica", r"mas", r"iras", r"hdb",
            r"officer", r"detective", r"warrant", r"arrest", r"investigation",
            r"money laundering", r"safety account", r"court", r"ministry",
            r"government", r"official", r"authority", r"polis",
            r"警察", r"公安", r"调查", r"安全账户",
            r"காவல்", r"அரசு",
        ],
        "advice": (
            "Cases MORE THAN DOUBLED in 2025 (+123.6%), $242.9M lost (SPF 2025). "
            "Singapore government officials will NEVER ask you to transfer money, "
            "disclose banking details, or install unofficial apps. "
            "Hang up and call ScamShield at 1799."
        ),
    },
    "bank_impersonation": {
        "spf_name"    : "Bank Impersonation Scam (SPF-tracked under Phishing Scam)",
        "2025_cases"  : 6264,     # borrowed from Phishing Scam -- see note above
        "2025_losses" : 39.9,
        "avg_loss"    : 6384,
        "risk_tier"   : "HIGH",
        "risk_score"  : 76,
        "keywords"    : [
            r"dbs", r"posb", r"ocbc", r"uob", r"standard chartered",
            r"citibank", r"hsbc", r"maybank", r"trust bank",
            r"security team", r"new device login", r"unauthorised access",
            r"account frozen", r"account suspended", r"security alert",
            r"kata laluan bank", r"akaun bank",
            r"银行", r"账户冻结", r"安全团队",
        ],
        "advice": (
            "SPF does not track bank impersonation as a standalone category — it "
            "is folded into Phishing Scam statistics ($39.9M lost, 6,264 cases, "
            "SPF 2025), where fake bank alerts push victims to fake login pages. "
            "Banks never ask you to verify your account via a link in an SMS or "
            "email. Go directly to your bank's official app. Report to "
            "report@scamalert.sg."
        ),
    },
    "job_scam": {
        "spf_name"    : "Job Scam",
        "2025_cases"  : 5575,
        "2025_losses" : 123.5,
        "avg_loss"    : 22163,
        "risk_tier"   : "HIGH",
        "risk_score"  : 80,
        "keywords"    : [
            r"job", r"work from home", r"part.?time", r"hiring", r"salary",
            r"task", r"commission", r"earn", r"vacancy", r"recruit",
            r"registration fee", r"training fee", r"deposit", r"agent fee",
            r"kerja", r"gaji", r"pendapatan",
            r"வேலை", r"சம்பளம்",
            r"工作", r"兼职", r"佣金", r"招聘",
        ],
        "advice": (
            "Job scams cost Singaporeans $123.5M in 2025 (SPF 2025). "
            "Legitimate employers never ask for upfront fees. "
            "Verify at mom.gov.sg. Report to MOM or call 1800-255-0000."
        ),
    },
    "phishing": {
        "spf_name"    : "Phishing Scam",
        "2025_cases"  : 6264,
        "2025_losses" : 39.9,
        "avg_loss"    : 6384,
        "risk_tier"   : "HIGH",
        "risk_score"  : 78,
        "keywords"    : [
            r"click", r"link", r"verify", r"otp", r"credential", r"login",
            r"password", r"http", r"www\.", r"\.xyz", r"\.top", r"\.club",
            r"kata laluan", r"akaun",
            r"கணக்கு", r"கடவுச்சொல்",
            r"账户", r"密码", r"验证", r"点击",
        ],
        "advice": (
            "Phishing is the 2nd most common scam in Singapore (6,264 cases, SPF 2025). "
            "Never click unsolicited links or enter OTP/password from an SMS or email. "
            "Go directly to your bank's official website. "
            "Report to report@scamalert.sg."
        ),
    },
    "ecommerce": {
        "spf_name"    : "E-commerce Scam",
        "2025_cases"  : 6703,
        "2025_losses" : 16.7,
        "avg_loss"    : 2503,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 60,
        "keywords"    : [
            r"sell", r"selling", r"buy", r"cheap", r"deal", r"item",
            r"carousell", r"shopee", r"lazada", r"facebook marketplace",
            r"ship", r"delivery", r"transfer first", r"paynow first",
            r"deposit", r"pokemon", r"brand new", r"legit",
            r"jual", r"beli", r"murah",
            r"விற்பனை", r"வாங்க",
            r"出售", r"购买", r"便宜", r"转账",
        ],
        "advice": (
            "E-commerce scams are the most common type in Singapore (6,703 cases, SPF 2025). "
            "Never PayNow before receiving goods. "
            "Meet in person for high-value items or use Carousell protected payment. "
            "Report at go.gov.sg/scamalert."
        ),
    },
    "fake_friend": {
        "spf_name"    : "Fake Friend Call Scam",
        "2025_cases"  : 1551,
        "2025_losses" : 4.7,
        "avg_loss"    : 3056,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 62,
        "keywords"    : [
            r"new number", r"changed number", r"it's me", r"lost my phone",
            r"lost phone", r"stuck", r"stranded", r"emergency",
            r"transfer.{0,15}(now|urgent|first)", r"pay you back",
            r"hi mum", r"hi dad", r"hi mom",
            r"妈妈", r"爸爸", r"新号码",
            r"அம்மா", r"அப்பா",
        ],
        "advice": (
            "Fake friend call scams fell sharply in 2025 but still cost $4.7M "
            "across 1,551 cases (SPF 2025). Always verify a claimed new number by "
            "calling the person's old number or contacting them another way "
            "before sending money."
        ),
    },
    "parcel_delivery": {
        "spf_name"    : "Parcel Delivery Scam",
        "2025_cases"  : None,   # not separately tracked in SPF Annual Brief 2025
        "2025_losses" : None,
        "avg_loss"    : None,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [
            r"parcel", r"delivery", r"courier", r"singpost", r"ninja van",
            r"dhl", r"fedex", r"j&t", r"tracking", r"customs",
            r"clearance fee", r"shipping fee", r"cannot deliver",
            r"包裹", r"快递", r"清关费",
            r"பார்சல்", r"விநியோகம்",
        ],
        "advice": (
            "SPF does not publish separate 2025 statistics for parcel delivery "
            "scams — a related advisory (Sept 2024) recorded 338+ cases and "
            "$616K+ in losses, largely impersonating SingPost. Legitimate couriers "
            "never request payment via SMS link for customs clearance. Verify "
            "directly with the courier's official app or hotline."
        ),
    },
    "rental": {
        "spf_name"    : "Rental Scam",
        "2025_cases"  : None,
        "2025_losses" : None,
        "avg_loss"    : None,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [
            r"rent", r"deposit", r"landlord", r"room", r"apartment",
            r"condo", r"hdb.{0,10}(rent|room|unit)", r"lease", r"reserve.{0,10}unit",
            r"租房", r"房租", r"押金",
            r"sewa", r"வாடகை",
        ],
        "advice": (
            "SPF does not publish separate 2025 statistics for rental scams; they "
            "are grouped within the SPF's broader 'Other Scams' bucket. Never "
            "transfer a deposit before viewing the property in person and "
            "verifying the landlord owns the unit (check against HDB/URA records)."
        ),
    },
    "loan": {
        "spf_name"    : "Loan Scam",
        "2025_cases"  : 935,
        "2025_losses" : 7.0,
        "avg_loss"    : 7515,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 64,
        "keywords"    : [
            r"loan", r"credit", r"approve", r"interest", r"fast cash",
            r"money lender", r"licensed moneylender", r"no credit check",
            r"emergency cash", r"disbursement",
            r"pinjaman", r"贷款", r"கடன்",
        ],
        "advice": (
            "Loan scams: 935 cases, $7.0M lost in 2025 (SPF 2025) — one of the few "
            "scam types where average loss rose despite fewer cases. Never pay "
            "upfront 'processing fees' before receiving a loan. Verify licensed "
            "moneylenders at Ministry of Law's registry (mlaw.gov.sg)."
        ),
    },
    "charity": {
        "spf_name"    : "Charity Scam",
        "2025_cases"  : None,
        "2025_losses" : None,
        "avg_loss"    : None,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [
            r"donate", r"donation", r"charity", r"fundraising",
            r"disaster relief", r"food bank", r"treatment fund",
            r"捐款", r"慈善", r"derma", r"நன்கொடை",
        ],
        "advice": (
            "SPF does not publish separate 2025 statistics for charity scams; "
            "they fall within the 'Other Scams' bucket. Donate only through "
            "registered charities listed on the Charity Portal (charities.gov.sg) "
            "and avoid urgent, unsolicited donation requests via messaging apps."
        ),
    },
    "prize": {
        "spf_name"    : "Prize Scam",
        "2025_cases"  : None,
        "2025_losses" : None,
        "avg_loss"    : None,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [
            r"winner", r"won", r"congratulations", r"lucky draw",
            r"claim.{0,10}prize", r"cash prize", r"ntuc voucher", r"reward",
            r"中奖", r"奖金", r"hadiah", r"பரிசு",
        ],
        "advice": (
            "SPF does not publish separate 2025 statistics for prize scams; they "
            "fall within the 'Other Scams' bucket. Legitimate prizes never "
            "require payment, personal banking details, or an OTP to claim."
        ),
    },
    "unknown": {
        "spf_name"    : "Other Scam",
        "2025_cases"  : 9941,
        "2025_losses" : 135.1,
        "avg_loss"    : 13590,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [],
        "advice": (
            "This message shows scam indicators. Do not send money, share personal "
            "details or OTPs, or click unknown links. "
            "Verify via ScamShield app or call 1799."
        ),
    },
}

def classify_scam_type(text: str) -> tuple:
    """Match text against SPF taxonomy keywords. Returns (scam_type, risk_score, risk_tier, advice)."""
    text_lower = text.lower()
    scores = {}
    for stype, info in SPF_TAXONOMY.items():
        if stype == "unknown":
            continue
        hits = sum(1 for kw in info["keywords"] if re.search(kw, text_lower))
        if hits > 0:
            scores[stype] = hits
    if not scores:
        t = SPF_TAXONOMY["unknown"]
        return "unknown", t["risk_score"], t["risk_tier"], t["advice"]
    best = max(scores, key=scores.get)
    t = SPF_TAXONOMY[best]
    return best, t["risk_score"], t["risk_tier"], t["advice"]

# Quick test
def show_taxonomy_result(text):
    scam_type, risk_score, risk_tier, advice = classify_scam_type(text)
    info = SPF_TAXONOMY[scam_type]

    print(f"INPUT: {text}")

    summary = pd.DataFrame([{
        "Detected Type": info["spf_name"],
        "Taxonomy Key": scam_type,
        "Risk Tier": risk_tier,
        "Risk Score": f"{risk_score}/100",
        "SPF Cases": (
            f"{info['2025_cases']:,}"
            if info["2025_cases"] is not None else "N/A"
        ),
        "Total Losses": (
            f"${info['2025_losses']}M"
            if info["2025_losses"] is not None else "N/A"
        ),
        "Avg Loss": (
            f"${info['avg_loss']:,}"
            if info["avg_loss"] is not None else "N/A"
        )
    }])

    display(summary.style.hide(axis="index"))

    print("\nADVICE: ")
    print(advice)
    print()

show_taxonomy_result(
    "guaranteed 300% returns on crypto investment!"
)

show_taxonomy_result(
    "Hi Mum, it's me, new number, please transfer $500 urgently"
)

print("SPF taxonomy ready — 12 categories aligned with NB01 synthetic labels + unknown fallback")

INPUT: guaranteed 300% returns on crypto investment!


Detected Type,Taxonomy Key,Risk Tier,Risk Score,SPF Cases,Total Losses,Avg Loss
Investment Scam,investment,CRITICAL,95/100,"5,462",$336.2M,"$61,559"



ADVICE: 
Investment scams caused the HIGHEST losses in Singapore in 2025 — $336.2 million (SPF 2025). Never transfer money to strangers for 'investments'. Verify at MAS (mas.gov.sg/investor-alert-list). Report to SPF at 1800-255-0000.

INPUT: Hi Mum, it's me, new number, please transfer $500 urgently


Detected Type,Taxonomy Key,Risk Tier,Risk Score,SPF Cases,Total Losses,Avg Loss
Fake Friend Call Scam,fake_friend,MEDIUM,62/100,"1,551",$4.7M,"$3,056"



ADVICE: 
Fake friend call scams fell sharply in 2025 but still cost $4.7M across 1,551 cases (SPF 2025). Always verify a claimed new number by calling the person's old number or contacting them another way before sending money.

SPF taxonomy ready — 12 categories aligned with NB01 synthetic labels + unknown fallback


## Cell 5 — SPF Advisory Corpus + FAISS Index

The SPF corpus contains **30 advisory passages** extracted from the SPF 2025 brief.
Each passage is a self-contained chunk covering one scam type or statistic.

These are embedded using `paraphrase-multilingual-MiniLM-L12-v2` (supports all 5 languages)
and stored in a FAISS index for fast cosine-similarity retrieval.

The index is saved to disk so it only needs to be built once per Kaggle session.


In [15]:
# SPF advisory corpus for RAG retrieval. scam_type values are aligned 1:1 with
# SPF_TAXONOMY keys (Cell 4) and NB01's synthetic labels, so retrieve_spf_passages()
# always finds a type-matched passage for every category the classifier can return.
SPF_CORPUS = [
    # ── OVERALL ──────────────────────────────────────────────────────────────
    {"id":"spf_001","scam_type":"overview","topic":"Overall scam situation 2025","source_page":1,
     "text":("In 2025, scam and cybercrime cases decreased by 24.8% to 41,974 cases. "
             "Scam cases fell by 27.6% to 37,308 cases. Total losses fell 17.9% to $913.1 million. "
             "Despite the decrease, the situation remains very concerning.")},
    {"id":"spf_002","scam_type":"overview","topic":"Self-effected transfers 2025","source_page":4,
     "text":("81.8% of scams involved self-effected transfers — scammers manipulated victims into "
             "performing monetary transactions through deception and social engineering, "
             "without gaining direct control of victims' accounts.")},
    {"id":"spf_003","scam_type":"overview","topic":"Cryptocurrency losses 2025","source_page":3,
     "text":("Cryptocurrency losses accounted for $182.2 million (about 20% of total scam losses) "
             "in 2025. Tether, Ethereum, and Bitcoin accounted for 91.7% of cryptocurrency losses. "
             "Crypto transactions are irreversible, making recovery very challenging.")},
    {"id":"spf_004","scam_type":"overview","topic":"Top scam types by cases 2025","source_page":4,
     "text":("Top 5 scam types by cases in 2025: e-commerce (6,703 cases, 18.0%), "
             "phishing (6,264 cases, 16.8%), job scams (5,575 cases, 14.9%), "
             "investment scams (5,462 cases, 14.6%), government impersonation (3,363 cases, 9.0%).")},
    {"id":"spf_005","scam_type":"overview","topic":"Top scam types by losses 2025","source_page":5,
     "text":("Top 5 scam types by losses in 2025: investment scams ($336.2M, 36.8%), "
             "government impersonation ($242.9M, 26.6%), job scams ($123.5M, 13.5%), "
             "phishing ($39.9M, 4.4%), business email compromise ($35.3M, 3.9%).")},
    # ── INVESTMENT ────────────────────────────────────────────────────────────
    {"id":"spf_006","scam_type":"investment","topic":"Investment scam statistics 2025","source_page":7,
     "text":("Investment scams recorded the highest losses in 2025: $336.2 million (+4.8% from 2024). "
             "There were 5,462 cases. Average loss per victim: $61,559 — highest of all scam types.")},
    {"id":"spf_007","scam_type":"investment","topic":"Investment scam tactics — platforms","source_page":18,
     "text":("Victims encountered investment opportunities via Telegram and Facebook chat groups, "
             "online ads, and recommendations from online contacts. They were shown false testimonies "
             "and instructed to transfer money to bank accounts or cryptocurrency wallets.")},
    {"id":"spf_008","scam_type":"investment","topic":"Investment scam — crypto tactics","source_page":18,
     "text":("Investment scams accounted for 38.4% of all crypto-related scam cases in 2025. "
             "Victims were directed to open crypto accounts, fund them from bank accounts, "
             "then transfer cryptocurrency to scammer-controlled wallets.")},
    {"id":"spf_009","scam_type":"investment","topic":"Investment scam — fake apps","source_page":19,
     "text":("Scammers directed victims to download fake investment apps showing fictitious profits. "
             "Victims only discovered the scam when they attempted to withdraw returns and "
             "were asked to pay increasingly large 'fees' or 'taxes'.")},
    # ── GOVERNMENT IMPERSONATION ────────────────────────────────────────────────
    {"id":"spf_010","scam_type":"government_impersonation","topic":"Government impersonation statistics 2025","source_page":6,
     "text":("Government impersonation scams more than doubled in 2025 (+123.6% to 3,363 cases). "
             "Losses were $242.9 million (+60.5%). Average loss per victim: $72,229 — highest of all types.")},
    {"id":"spf_011","scam_type":"government_impersonation","topic":"Government impersonation — bank transfer tactic","source_page":17,
     "text":("91.7% of government impersonation cases: victims received unsolicited calls from scammers "
             "posing as bank representatives, then were transferred to fake government officials who "
             "accused them of money laundering and told them to transfer funds to 'safety accounts'.")},
    {"id":"spf_012","scam_type":"government_impersonation","topic":"What Singapore government officials will never do","source_page":18,
     "text":("Singapore Government officials will NEVER ask you over a phone call to: transfer money, "
             "disclose banking login details, install apps from unofficial stores, or transfer "
             "your call to the Police. Never hand money or valuables to any unknown person.")},
    {"id":"spf_013","scam_type":"government_impersonation","topic":"Government impersonation — PayNow and crypto","source_page":17,
     "text":("New 2025 tactics: scammers instructed victims to transfer funds via PayNow to "
             "Payment Service Provider accounts (e.g. YouTrip), or to open new crypto accounts "
             "and transfer cryptocurrency directly to scammer-controlled wallets.")},
    # ── BANK IMPERSONATION (folded into SPF's Phishing Scam category) ──────────
    {"id":"spf_031","scam_type":"bank_impersonation","topic":"Bank impersonation as a phishing sub-pattern","source_page":7,
     "text":("SPF does not track bank impersonation as a standalone scam type; it is captured within "
             "Phishing Scam statistics. Victims receive fake alerts claiming their bank account is "
             "frozen, suspended, or flagged, and are directed to a fraudulent login page to 'verify' "
             "their identity, resulting in credential or card theft.")},
    {"id":"spf_032","scam_type":"bank_impersonation","topic":"Bank impersonation — never share OTP","source_page":7,
     "text":("Banks will never ask a customer to verify their account, provide an OTP, or confirm login "
             "credentials via a link sent by SMS or email. Genuine account issues are addressed by "
             "logging in directly through the bank's official app, not through a message link.")},
    # ── JOB SCAMS ─────────────────────────────────────────────────────────────
    {"id":"spf_014","scam_type":"job_scam","topic":"Job scam statistics 2025","source_page":16,
     "text":("Job scams: 5,575 cases in 2025 (down 38.4% from 2024). Total losses: $123.5 million. "
             "Average loss per victim: $22,163. Third highest by both cases and losses.")},
    {"id":"spf_015","scam_type":"job_scam","topic":"Job scam tactics — upfront fees and fake tasks","source_page":16,
     "text":("Job scammers advertise easy work-from-home jobs with high pay on social media. "
             "Victims are asked to pay registration fees, training fees, or deposits before starting. "
             "Some involve fake online tasks (liking posts, boosting products) with small initial payments "
             "to build trust, before large sums are requested and never returned.")},
    {"id":"spf_016","scam_type":"job_scam","topic":"Job scam — victim demographics","source_page":9,
     "text":("Job scams disproportionately targeted young adults aged 20–29 (19.9% of all victims) "
             "and adults aged 30–49 (17.6% falling prey to job scams). "
             "Legitimate employers do not ask for upfront fees before employment.")},
    # ── PHISHING ──────────────────────────────────────────────────────────────
    {"id":"spf_017","scam_type":"phishing","topic":"Phishing scam statistics 2025","source_page":7,
     "text":("Phishing scams: 6,264 cases in 2025 (down 26.8% from 2024). "
             "Total losses: $39.9 million (down 32.8%). Average loss per victim: $6,384.")},
    {"id":"spf_018","scam_type":"phishing","topic":"Phishing tactics — card details and OTP theft","source_page":7,
     "text":("Phishing scams predominantly involved victims submitting card details and OTPs "
             "on fake websites impersonating banks, government agencies, or e-commerce platforms, "
             "resulting in unauthorised card transactions.")},
    {"id":"spf_019","scam_type":"phishing","topic":"Phishing contact methods","source_page":11,
     "text":("Phishing scammers contact victims via SMS, email, and messaging platforms with links "
             "to fraudulent websites. Major retail banks have phased out SMS OTPs for digital token users. "
             "Never click links in unsolicited messages — go directly to your bank's official app or website.")},
    # ── E-COMMERCE ────────────────────────────────────────────────────────────
    {"id":"spf_020","scam_type":"ecommerce","topic":"E-commerce scam statistics 2025","source_page":6,
     "text":("E-commerce scams: 6,703 cases in 2025 (down 42.5% from 2024, but still the most cases). "
             "Total losses: $16.7 million. Average loss per victim: $2,503.")},
    {"id":"spf_021","scam_type":"ecommerce","topic":"E-commerce scam platforms and payment","source_page":6,
     "text":("Carousell (29.0%) and Facebook Marketplace (22.2%) were the most used platforms. "
             "Victims were asked to pay via PayNow or bank transfer upfront. "
             "They discovered the scam when goods were not delivered and sellers became uncontactable.")},
    {"id":"spf_022","scam_type":"ecommerce","topic":"E-commerce scam — most common items","source_page":6,
     "text":("Pokemon trading cards were the most commonly scammed item (13.6% of e-commerce cases). "
             "Other items: electronics, concert tickets, luxury goods. "
             "Use platforms' protected payment and meet sellers in person for high-value items.")},
    # ── FAKE FRIEND CALL SCAM (stats-only — no Annex D narrative in SPF report) ─
    {"id":"spf_033","scam_type":"fake_friend","topic":"Fake friend call scam statistics 2025","source_page":16,
     "text":("Fake friend call scams: 1,551 cases in 2025, down sharply from 4,179 cases in 2024. "
             "Total losses fell to $4.7 million, from $13.6 million in 2024. "
             "Average loss per victim: $3,056. One of the scam types with the largest year-on-year decrease.")},
    {"id":"spf_034","scam_type":"fake_friend","topic":"Fake friend call scam — verify before transferring","source_page":16,
     "text":("Scammers claiming to be a friend or family member with a 'new number' typically request "
             "urgent money transfers citing an emergency, without giving the victim time to verify. "
             "Always call the person's known number, or contact them through another channel, before "
             "transferring any money.")},
    # ── LOAN SCAM (stats-only — no Annex D narrative in SPF report) ─────────────
    {"id":"spf_035","scam_type":"loan","topic":"Loan scam statistics 2025","source_page":16,
     "text":("Loan scams: 935 cases in 2025, down from 1,154 cases in 2024. "
             "Total losses rose to $7.0 million, from $6.0 million in 2024 — one of the few scam types "
             "where losses increased despite fewer cases. Average loss per victim: $7,515.")},
    {"id":"spf_036","scam_type":"loan","topic":"Loan scam — verify licensed moneylenders","source_page":16,
     "text":("Legitimate licensed moneylenders never guarantee approval without checks, and never "
             "request an upfront 'processing' or 'unlocking' fee before disbursing a loan. Verify any "
             "moneylender against the Ministry of Law's registry before proceeding.")},
    # ── PARCEL DELIVERY SCAM (no official 2025 SPF stats — separate advisory only) ─
    {"id":"spf_037","scam_type":"parcel_delivery","topic":"Parcel delivery phishing — not separately tracked in 2025 Brief","source_page":None,
     "text":("Parcel delivery scams are not broken out separately in the SPF Annual Scam and Cybercrime "
             "Brief 2025; they fall within the broader 'Other Scams' bucket. A dedicated SPF advisory "
             "(September 2024) recorded 338+ cases and $616,000+ in losses since January 2024, mostly "
             "impersonating SingPost with fake failed-delivery messages.")},
    {"id":"spf_038","scam_type":"parcel_delivery","topic":"Parcel delivery scam tactic","source_page":None,
     "text":("Victims receive a message claiming a parcel delivery failed and are asked to click a link "
             "to 'confirm their address' or pay a small clearance fee. The link leads to a phishing page "
             "that harvests card details rather than a legitimate courier site.")},
    # ── RENTAL SCAM (no official SPF stats) ─────────────────────────────────────
    {"id":"spf_039","scam_type":"rental","topic":"Rental scam — not separately tracked in 2025 Brief","source_page":None,
     "text":("Rental scams are not broken out separately in the SPF Annual Scam and Cybercrime Brief "
             "2025; they fall within the broader 'Other Scams' bucket. SPF enforcement operations in "
             "late 2025 noted rental scams among the mix of case types investigated alongside "
             "e-commerce and impersonation scams.")},
    {"id":"spf_040","scam_type":"rental","topic":"Rental scam tactic","source_page":None,
     "text":("Scammers advertise attractively priced rooms or units and pressure prospective tenants "
             "to transfer a deposit quickly, citing high demand, before any viewing takes place. "
             "Never transfer a deposit before viewing the property and verifying ownership.")},
    # ── CHARITY SCAM (no official SPF stats) ────────────────────────────────────
    {"id":"spf_041","scam_type":"charity","topic":"Charity scam — not separately tracked in 2025 Brief","source_page":None,
     "text":("Charity scams are not broken out separately in the SPF Annual Scam and Cybercrime Brief "
             "2025; they fall within the broader 'Other Scams' bucket. Donate only through registered "
             "charities listed on the Charity Portal, and be wary of urgent, unsolicited donation "
             "requests received via messaging apps.")},
    # ── PRIZE SCAM (no official SPF stats) ──────────────────────────────────────
    {"id":"spf_042","scam_type":"prize","topic":"Prize scam — not separately tracked in 2025 Brief","source_page":None,
     "text":("Prize scams are not broken out separately in the SPF Annual Scam and Cybercrime Brief "
             "2025; they fall within the broader 'Other Scams' bucket. Legitimate lucky draws and "
             "prizes never require payment, banking details, or an OTP to release winnings.")},
    # ── CONTACT METHODS ───────────────────────────────────────────────────────
    {"id":"spf_025","scam_type":"overview","topic":"Online platforms used by scammers 2025","source_page":8,
     "text":("Scammers used online platforms in 84.1% of all cases. Meta platforms: 35.4% of cases. "
             "Top contact methods: social media (10,448 cases), messaging platforms (9,355), "
             "phone calls (5,477), online shopping platforms (3,804).")},
    {"id":"spf_026","scam_type":"overview","topic":"WhatsApp and Telegram most used for scam contact","source_page":8,
     "text":("Among messaging platforms: WhatsApp 53.5%, Telegram 37.9%, Facebook Messenger 4.2%. "
             "Among social media: Facebook 51.9%, TikTok 26.0%, Instagram 14.2% of scam contact cases.")},
    # ── VICTIM PROFILE ────────────────────────────────────────────────────────
    {"id":"spf_027","scam_type":"overview","topic":"Scam victim age profile 2025","source_page":9,
     "text":("85.2% of scam victims were aged below 65. Adults aged 30–49: 36.1% of victims ($22,283 avg loss). "
             "Elderly aged 65+: 14.8% with the highest average loss of $37,053, mainly from investment, "
             "impersonation, and phishing scams.")},
    # ── ANTI-SCAM ACTIONS ─────────────────────────────────────────────────────
    {"id":"spf_028","scam_type":"overview","topic":"ScamShield helpline and reporting channels","source_page":13,
     "text":("When in doubt, call ScamShield Helpline at 1799 or use the ScamShield app. "
             "Report scams to SPF at 1800-255-0000 or police.gov.sg/i-witness. "
             "Report phishing links to report@scamalert.sg. "
             "Check investment legitimacy at MAS Investor Alert List: mas.gov.sg.")},
    {"id":"spf_029","scam_type":"overview","topic":"Anti-scam laws — caning and Protection from Scams Act","source_page":10,
     "text":("From 30 December 2025, caning for scam offences was operationalised (6–24 strokes). "
             "The Protection from Scams Act (1 July 2025) allows SPF to restrict banking facilities "
             "of scam victims who continue to believe scammers despite police engagement.")},
    {"id":"spf_030","scam_type":"overview","topic":"Money mules — do not share accounts or Singpass","source_page":13,
     "text":("Do not allow anyone to use your financial accounts to transfer funds, share Singpass "
             "credentials, or supply local SIM cards. In 2025, 7,000+ money mules were investigated "
             "and 940+ charged. Penalties include substantial prison terms and caning.")},
]

print(f"SPF corpus loaded: {len(SPF_CORPUS)} advisory chunks")
print(f"Scam types covered: {sorted(set(c['scam_type'] for c in SPF_CORPUS))}")


SPF corpus loaded: 40 advisory chunks
Scam types covered: ['bank_impersonation', 'charity', 'ecommerce', 'fake_friend', 'government_impersonation', 'investment', 'job_scam', 'loan', 'overview', 'parcel_delivery', 'phishing', 'prize', 'rental']


In [16]:
# ── Build or load FAISS index ─────────────────────────────────────────────────
INDEX_PATH     = RAG_DIR / "spf_faiss.index"
EMBEDDINGS_PATH = RAG_DIR / "spf_embeddings.npy"

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

if INDEX_PATH.exists() and EMBEDDINGS_PATH.exists():
    index             = faiss.read_index(str(INDEX_PATH))
    corpus_embeddings = np.load(str(EMBEDDINGS_PATH))
    print(f"FAISS index loaded from disk ({index.ntotal} vectors)")
else:
    print("Building FAISS index from SPF corpus ...")
    corpus_texts      = [c["text"] for c in SPF_CORPUS]
    corpus_embeddings = embedder.encode(
        corpus_texts,
        normalize_embeddings=True,   
        show_progress_bar=True
    ).astype(np.float32)

    DIM   = corpus_embeddings.shape[1]
    index = faiss.IndexFlatIP(DIM)
    index.add(corpus_embeddings)

    faiss.write_index(index, str(INDEX_PATH))
    np.save(str(EMBEDDINGS_PATH), corpus_embeddings)
    print(f"FAISS index built ({index.ntotal} vectors, dim={DIM})")
    print(f"   Saved → {INDEX_PATH}")

# Quick retrieval test
q = embedder.encode(["phishing bank account OTP"], normalize_embeddings=True).astype(np.float32)
scores, idxs = index.search(q, 3)
print("\nRetrieval test (phishing query):")
for s, i in zip(scores[0], idxs[0]):
    print(f"  [{s:.3f}] {SPF_CORPUS[i]['id']} — {SPF_CORPUS[i]['topic']}")

FAISS index loaded from disk (40 vectors)

Retrieval test (phishing query):
  [0.721] spf_019 — Phishing contact methods
  [0.686] spf_018 — Phishing tactics — card details and OTP theft
  [0.537] spf_031 — Bank impersonation as a phishing sub-pattern


## Cell 6 — SHAP Explainer, Language Detector & Agent State Schema

In [17]:
# ── SHAP ─────────────────────────────────────────────────────────────────────
def shap_predict(texts):
    """Wrapper so SHAP can query the classifier. Returns [ham_prob, scam_prob] per text."""
    if isinstance(texts, str):
        texts = [texts]
    inputs = clf_tokenizer(
        list(texts), return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

masker    = shap.maskers.Text(clf_tokenizer)
explainer = shap.Explainer(shap_predict, masker, output_names=["ham", "scam"])

def get_top_shap_features(text: str, top_n: int = 5) -> list[dict]:
    """Return top_n tokens most responsible for the scam prediction (by SHAP value)."""
    shap_values = explainer([text])
    token_names = shap_values.data[0]
    token_shaps = shap_values.values[0, :, 1]   # scam class
    pairs = [
        {"token": tok, "shap_value": round(float(val), 4)}
        for tok, val in zip(token_names, token_shaps)
        if tok not in ["", "▁", "<s>", "</s>", "<pad>"]
    ]
    return sorted(pairs, key=lambda x: x["shap_value"], reverse=True)[:top_n]

print("SHAP explainer ready")

# ── Singlish heuristics ───────────────────────────────────────────────────────
SINGLISH_MARKERS = [
    r"\blah\b", r"\bleh\b", r"\bsia\b", r"\blor\b", r"\bwah\b",
    r"\baiyo\b", r"\bsiao\b", r"\bdun\b", r"\bwan\b", r"\bcan or not\b",
    r"\bgot meh\b", r"\bcan leh\b", r"\bone\b",
]
_singlish_re = re.compile("|".join(SINGLISH_MARKERS), re.IGNORECASE)

def detect_language(text: str) -> str:
    hits  = len(_singlish_re.findall(text))
    words = len(text.split())
    if words > 0 and (hits / words) >= 0.08:
        return "singlish"
    try:
        d = detect(text)
        return {"en":"en","ms":"ms","id":"ms","ta":"ta",
                "zh-cn":"zh","zh-tw":"zh","zh":"zh"}.get(d, "en")
    except LangDetectException:
        return "en"

print("Language detector ready")

# ── LangGraph AgentState ──────────────────────────────────────────────────────
class AgentState(TypedDict):
    # Input
    message      : str
    # Detection Agent outputs
    language     : Optional[str]
    label        : Optional[str]
    confidence   : Optional[float]
    scam_prob    : Optional[float]
    # Explanation Agent outputs
    top_features : Optional[list]
    rag_chunks   : Optional[list]   # retrieved SPF passages
    rag_summary  : Optional[str]    # condensed RAG explanation
    # Risk Agent outputs
    scam_type    : Optional[str]
    spf_name     : Optional[str]
    risk_score   : Optional[int]
    risk_tier    : Optional[str]
    advice       : Optional[str]
    # Final combined output
    output       : Optional[dict]

print("AgentState schema defined")

SHAP explainer ready
Language detector ready
AgentState schema defined


## Cell 7 — Define the Three Agent Nodes

Each agent is a **LangGraph node** — a function that receives the full shared state,
does its work, and returns updated state fields.

- **Detection Agent**: XLM-RoBERTa classification + language detection
- **Explanation Agent**: SHAP token attribution + **RAG retrieval from FAISS** (SPF corpus)
- **Risk Agent**: SPF taxonomy matching + risk score + advice


In [18]:
# ── RAG retrieval helper ──────────────────────────────────────────────────────
def retrieve_spf_passages(message: str, scam_type: str, top_k: int = 3) -> list[dict]:
    """
    Retrieve top-k relevant SPF advisory passages for a message.
    Combines scam-type-specific results with general top results, deduplicated.
    """
    q_embed = embedder.encode([message], normalize_embeddings=True).astype(np.float32)

    # Type-specific retrieval (higher precision)
    type_indices = [i for i, c in enumerate(SPF_CORPUS) if c["scam_type"] == scam_type]
    if not type_indices:
        type_indices = list(range(len(SPF_CORPUS)))

    type_embeds = corpus_embeddings[type_indices]
    type_scores = (q_embed @ type_embeds.T)[0]
    top_type    = np.argsort(type_scores)[::-1][:top_k]

    results, seen = [], set()
    for local_i in top_type:
        chunk = SPF_CORPUS[type_indices[local_i]].copy()
        chunk["score"] = round(float(type_scores[local_i]), 4)
        if chunk["id"] not in seen:
            seen.add(chunk["id"])
            results.append(chunk)

    # Add general top result for breadth
    gen_scores, gen_idx = index.search(q_embed, 2)
    for s, i in zip(gen_scores[0], gen_idx[0]):
        chunk = SPF_CORPUS[i].copy()
        chunk["score"] = round(float(s), 4)
        if chunk["id"] not in seen:
            seen.add(chunk["id"])
            results.append(chunk)

    return results[:top_k + 1]


# ─────────────────────────────────────────────────────────────────────────────
# Agent 1: Detection Agent
# ─────────────────────────────────────────────────────────────────────────────
def detection_agent(state: AgentState) -> AgentState:
    """
    Detects language and classifies the message as scam or ham using XLM-RoBERTa.
    Populates: language, label, confidence, scam_prob.
    """
    lang   = detect_language(state["message"])
    result = run_classifier(state["message"])
    return {
        **state,
        "language"  : lang,
        "label"     : result["label"],
        "confidence": result["confidence"],
        "scam_prob" : result["scam_prob"],
    }


# ── Conditional routing after Detection Agent ─────────────────────────────────
def route_after_detection(state: AgentState) -> str:
    """Skip Explanation + Risk agents if message is clearly safe (ham ≥ 80% confidence)."""
    if state["label"] == "ham" and state["confidence"] >= 0.80:
        return "safe_exit"
    return "explanation_agent"


# ─────────────────────────────────────────────────────────────────────────────
# Agent 2: Explanation Agent (SHAP + RAG)
# ─────────────────────────────────────────────────────────────────────────────
def explanation_agent(state: AgentState) -> AgentState:
    """
    Uses SHAP to identify suspicious tokens, then retrieves relevant SPF advisory
    passages from the FAISS index to ground the explanation in real SPF data.

    This is the only agent that uses RAG.
    Populates: top_features, rag_chunks, rag_summary.
    """
    # SHAP: which tokens pushed the model toward 'scam'?
    top_features = get_top_shap_features(state["message"], top_n=5)

    # Determine likely scam type early (for targeted RAG retrieval)
    scam_type_guess, _, _, _ = classify_scam_type(state["message"])

    # RAG: retrieve most relevant SPF passages
    rag_chunks = retrieve_spf_passages(state["message"], scam_type_guess, top_k=3)

    # Build a concise RAG-grounded summary from retrieved passages
    key_tokens = ", ".join(f"'{f['token']}'" for f in top_features[:3]) if top_features else "—"
    best_chunk = rag_chunks[0] if rag_chunks else None

    if best_chunk:
        page_label = f"p.{best_chunk['source_page']}" if best_chunk.get('source_page') else "advisory (page n/a)"
        rag_summary = (
            f"Key suspicious tokens: {key_tokens}. "
            f"SPF 2025 ({page_label}): {best_chunk['text'][:220].rstrip()}..."
        )
    else:
        rag_summary = f"Key suspicious tokens: {key_tokens}. No SPF passage retrieved."

    return {
        **state,
        "top_features": top_features,
        "rag_chunks"  : rag_chunks,
        "rag_summary" : rag_summary,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Agent 3: Risk Agent
# ─────────────────────────────────────────────────────────────────────────────
def risk_agent(state: AgentState) -> AgentState:
    """
    Maps the message to the SPF 2025 taxonomy, computes a risk score,
    and selects an appropriate advisory.
    Populates: scam_type, spf_name, risk_score, risk_tier, advice.
    """
    scam_type, base_score, risk_tier, advice = classify_scam_type(state["message"])

    
    adjusted_score = max(10, min(100, int(base_score * state["scam_prob"])))

    if adjusted_score >= 85:
        risk_tier = "CRITICAL"
    elif adjusted_score >= 65:
        risk_tier = "HIGH"
    elif adjusted_score >= 40:
        risk_tier = "MEDIUM"
    else:
        risk_tier = "LOW"

    spf_info = SPF_TAXONOMY.get(scam_type, SPF_TAXONOMY["unknown"])

    output = {
        "message"         : state["message"],
        "language"        : state["language"],
        "label"           : state["label"],
        "confidence"      : state["confidence"],
        "scam_prob"       : state["scam_prob"],
        "top_features"    : state.get("top_features", []),
        "rag_chunks"      : state.get("rag_chunks", []),
        "rag_summary"     : state.get("rag_summary", ""),
        "scam_type"       : scam_type,
        "spf_name"        : spf_info["spf_name"],
        "spf_cases_2025"  : spf_info["2025_cases"],
        "spf_losses_2025" : spf_info["2025_losses"],
        "avg_loss_sgd"    : spf_info["avg_loss"],
        "risk_score"      : adjusted_score,
        "risk_tier"       : risk_tier,
        "advice"          : advice,
    }
    return {**state, "scam_type": scam_type, "spf_name": spf_info["spf_name"],
            "risk_score": adjusted_score, "risk_tier": risk_tier,
            "advice": advice, "output": output}


# ── Safe exit for clearly legitimate messages ─────────────────────────────────
def safe_exit_node(state: AgentState) -> AgentState:
    output = {
        "message"         : state["message"],
        "language"        : state["language"],
        "label"           : "ham",
        "confidence"      : state["confidence"],
        "scam_prob"       : state["scam_prob"],
        "top_features"    : [],
        "rag_chunks"      : [],
        "rag_summary"     : "",
        "scam_type"       : None,
        "spf_name"        : None,
        "spf_cases_2025"  : None,
        "spf_losses_2025" : None,
        "avg_loss_sgd"    : None,
        "risk_score"      : 0,
        "risk_tier"       : "NONE",
        "advice"          : "This message appears legitimate. No action needed.",
    }
    return {**state, "output": output}

print("All 3 agents + safe exit node defined")

All 3 agents + safe exit node defined


## Cell 8 — Assemble LangGraph State Machine

In [23]:
graph = StateGraph(AgentState)

# Register all nodes
graph.add_node("detection_agent",   detection_agent)
graph.add_node("explanation_agent", explanation_agent)
graph.add_node("risk_agent",        risk_agent)
graph.add_node("safe_exit",         safe_exit_node)

# Entry point
graph.set_entry_point("detection_agent")

# Conditional routing: ham with high confidence → skip to safe_exit
graph.add_conditional_edges(
    "detection_agent",
    route_after_detection,
    {
        "safe_exit"        : "safe_exit",
        "explanation_agent": "explanation_agent",
    }
)

# Scam path: Explanation → Risk → END
graph.add_edge("explanation_agent", "risk_agent")
graph.add_edge("risk_agent",        END)
graph.add_edge("safe_exit",         END)

pipeline = graph.compile()

print("LangGraph pipeline compiled")
print("Nodes:", list(pipeline.get_graph().nodes.keys()))

LangGraph pipeline compiled
Nodes: ['__start__', 'detection_agent', 'explanation_agent', 'risk_agent', 'safe_exit', '__end__']


## Cell 9 — Inference: Seven Test Cases (One per Language & Scam Type)

In [24]:
import pandas as pd
from IPython.display import display

TIER_ICONS = {
    "CRITICAL": "🔴",
    "HIGH": "🟠",
    "MEDIUM": "🟡",
    "LOW": "🟢",
    "NONE": "✅"
}

LANG_NAMES = {
    "en": "English",
    "zh": "Mandarin",
    "ta": "Tamil",
    "ms": "Malay",
    "singlish": "Singlish"
}


def _show(df):
    display(df.style.hide(axis="index"))

# Pipeline runner
def run_pipeline_display(name: str, message: str):

    initial: AgentState = {
        "message": message,
        "language": None,
        "label": None,
        "confidence": None,
        "scam_prob": None,
        "top_features": None,
        "rag_chunks": None,
        "rag_summary": None,
        "scam_type": None,
        "spf_name": None,
        "risk_score": None,
        "risk_tier": None,
        "advice": None,
        "output": None,
    }

    final = pipeline.invoke(initial)
    result = final["output"]

    lang = LANG_NAMES.get(result["language"], result["language"])
    tier = result.get("risk_tier", "NONE")
    icon = TIER_ICONS.get(tier, "❓")

    print(f"TEST CASE: {name.upper()}")
    print("-" * 90)
    print(f"INPUT: {message}")

    # SUMMARY
    print("\nSUMMARY")

    if result["label"] == "scam":

        summary = pd.DataFrame([{
            "Test Case": name.upper(),
            "Prediction": result["label"].upper(),
            "Confidence": f"{result['confidence']:.1%}",
            "Language": lang,
            "Scam Prob": f"{result['scam_prob']:.1%}",
            "Risk Tier": f"{icon} {tier}",
            "Risk Score": f"{result['risk_score']}/100",
            "Scam Type": result["spf_name"],
            "Tracked": "✅" if result.get("officially_tracked") else "—",
            "SPF Cases": (
                f"{result['spf_cases_2025']:,}"
                if result.get("spf_cases_2025") is not None else "N/A"
            ),
            "Total Losses": (
                f"${result['spf_losses_2025']}M"
                if result.get("spf_losses_2025") is not None else "N/A"
            ),
            "Avg Loss": (
                f"${result['avg_loss_sgd']:,}"
                if result.get("avg_loss_sgd") is not None else "N/A"
            )
        }])

    else:

        summary = pd.DataFrame([{
            "Test Case": name.upper(),
            "Prediction": result["label"].upper(),
            "Confidence": f"{result['confidence']:.1%}",
            "Language": lang,
            "Scam Prob": f"{result['scam_prob']:.1%}",
        }])

    _show(summary)

    # SHAP
    if result["label"] == "scam" and result.get("top_features"):

        print("\nTOP SHAP TOKENS")

        shap_tbl = pd.DataFrame([
            {
                "Rank": i + 1,
                "Token": f["token"],
                "SHAP Score": f"{f['shap_value']:+.4f}"
            }
            for i, f in enumerate(result["top_features"])
        ])

        _show(shap_tbl)

    # RAG
    if result["label"] == "scam" and result.get("rag_chunks"):

        print("\nRAG RETRIEVAL")

        rag_tbl = pd.DataFrame([
            {
                "ID": c["id"],
                "Type": c["scam_type"],
                "Score": f"{c['score']:.3f}",
                "Page": c["source_page"],
                "Topic": c["topic"]
            }
            for c in result["rag_chunks"]
        ])

        _show(rag_tbl)

    if result["label"] == "scam":

        print("\nEXPLANATION")
        print(result["rag_summary"])

    print("\nADVICE")
    print(result["advice"])

    return result

# Test cases
test_cases = [
    ("en / phishing",
     "URGENT: Your POSB account has been locked due to suspicious activity. Verify immediately at https://posb-secure-login.xyz using your OTP or your account will be suspended."),

    ("ta / investment",
     "300% லாபம் உறுதி! இன்று முதலீடு செய்து 30 நாட்களில் உங்கள் பணத்தை மூன்று மடங்காக பெறுங்கள். இப்போதே Telegram-ல் பதிவு செய்யுங்கள்."),

    ("ms / job_scam",
     "Kerja dari rumah! Pendapatan RM8,000 sebulan tanpa pengalaman. Bayar yuran pendaftaran RM300 sekarang untuk mula bekerja."),

    ("singlish / fake_friend",
     "Eh bro, this my new number lah. Wallet kena stolen overseas. Can PayNow me $500 first? Very urgent, I pay you back tomorrow confirm."),

    ("zh / government_impersonation",
     "您好，我是新加坡警察局警官。您的银行账户涉嫌洗钱，请立即将所有资金转入安全账户，否则今天会发出逮捕令。"),
    
    ("en / ham",
     "Hi John, your dental appointment is confirmed for tomorrow at 2:00 PM. Please arrive 10 minutes early."),

    ("ta / ham",
     "வணக்கம்! நாளை காலை 10 மணிக்கு பள்ளியில் பெற்றோர்-ஆசிரியர் சந்திப்பு நடைபெறும்."),

    ("ms / ham",
     "Hai! Mesyuarat projek akan bermula pada pukul 3 petang esok di Bilik Mesyuarat A. Sila hadir tepat pada waktunya."),

    ("singlish / ham",
     "Eh bro, we meeting at VivoCity 7pm tonight lah. Don't forget to bring your laptop for the project discussion."),

    ("zh / ham",
     "您好，提醒您明天下午2点在莱佛士医疗中心有预约。请提前10分钟到达，谢谢。"),
]

# Run
print("SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results")

all_results = []

for name, msg in test_cases:
    r = run_pipeline_display(name, msg)
    r["test_case"] = name
    all_results.append(r)

print("\nAll test cases complete.")

SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results
TEST CASE: EN / PHISHING
------------------------------------------------------------------------------------------
INPUT: URGENT: Your POSB account has been locked due to suspicious activity. Verify immediately at https://posb-secure-login.xyz using your OTP or your account will be suspended.

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob,Risk Tier,Risk Score,Scam Type,Tracked,SPF Cases,Total Losses,Avg Loss
EN / PHISHING,SCAM,100.0%,English,100.0%,🟠 HIGH,77/100,Phishing Scam,—,"6,264",$39.9M,"$6,384"



TOP SHAP TOKENS


Rank,Token,SHAP Score
1,lock,+0.0550
2,TP,+0.0401
3,cure,+0.0380
4,suspend,+0.0371
5,POS,+0.0363



RAG RETRIEVAL


...


ADVICE
This message appears legitimate. No action needed.
TEST CASE: SINGLISH / HAM
------------------------------------------------------------------------------------------
INPUT: Eh bro, we meeting at VivoCity 7pm tonight lah. Don't forget to bring your laptop for the project discussion.

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob
SINGLISH / HAM,HAM,100.0%,English,0.0%



ADVICE
This message appears legitimate. No action needed.
TEST CASE: ZH / HAM
------------------------------------------------------------------------------------------
INPUT: 您好，提醒您明天下午2点在莱佛士医疗中心有预约。请提前10分钟到达，谢谢。

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob
ZH / HAM,HAM,100.0%,Mandarin,0.0%



ADVICE
This message appears legitimate. No action needed.

All test cases complete.


## Cell 10 — Summary Table

In [25]:
import pandas as pd
from IPython.display import display

rows = []
for r in all_results:
    top_tok = r["top_features"][0]["token"] if r.get("top_features") else "—"
    rag_used = len(r.get("rag_chunks", []))

    rows.append({
        "Test Case": r["test_case"],
        "Prediction": r["label"].upper(),
        "Confidence": f"{r['confidence']:.1%}",
        "Language": LANG_NAMES.get(r["language"], r["language"]),
        "Scam Type": r.get("spf_name") or "—",
        "Risk Tier": f"{TIER_ICONS.get(r['risk_tier'], '')} {r['risk_tier']}",
        "Risk Score": r["risk_score"],
        "Top SHAP Token": top_tok,
        "RAG Passages": rag_used,
    })

df_summary = pd.DataFrame(rows)

print("  PIPELINE SUMMARY — ALL TEST CASES")

# Display as an interactive, scrollable table
display(df_summary)

# Save CSV
out_dir = BASE_DIR / "results"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "06_agent_inference_results.csv"
df_summary.to_csv(csv_path, index=False)

print(f"\nSummary table saved → {csv_path}")

  PIPELINE SUMMARY — ALL TEST CASES


,Test Case,Prediction,Confidence,Language,Scam Type,Risk Tier,Risk Score,Top SHAP Token,RAG Passages
0,en / phishing,SCAM,100.0%,English,Phishing Scam,🟠 HIGH,77,lock,4
1,ta / investment,SCAM,100.0%,Tamil,Investment Scam,🔴 CRITICAL,95,Telegram,4
2,ms / job_scam,SCAM,100.0%,Malay,Job Scam,🟠 HIGH,80,bekerja,3
3,singlish / fake_friend,SCAM,99.7%,English,Fake Friend Call Scam,🟡 MEDIUM,61,urgent,3
4,zh / government_impersonation,SCAM,99.6%,Mandarin,Government Officials Impersonation Scam,🔴 CRITICAL,92,您,3
5,en / ham,HAM,100.0%,English,—,✅ NONE,0,—,0
6,ta / ham,HAM,100.0%,Tamil,—,✅ NONE,0,—,0
7,ms / ham,HAM,99.9%,Malay,—,✅ NONE,0,—,0
8,singlish / ham,HAM,100.0%,English,—,✅ NONE,0,—,0
9,zh / ham,HAM,100.0%,Mandarin,—,✅ NONE,0,—,0



Summary table saved → /kaggle/working/ScamSense/results/06_agent_inference_results.csv


## Cell 11 — Save Artifacts & Push to GitHub

In [26]:
import json

# Save SPF taxonomy 
models_dir = BASE_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)

taxonomy_out = {
    k: {ek: ev for ek, ev in v.items() if ek != "keywords"}
    for k, v in SPF_TAXONOMY.items()
}
with open(models_dir / "spf_taxonomy.json", "w") as f:
    json.dump(taxonomy_out, f, indent=2)
print("SPF taxonomy saved → models/spf_taxonomy.json")

# Save SPF corpus JSON for downstream notebooks
import shutil
with open(RAG_DIR / "spf_corpus.json", "w") as f:
    json.dump(SPF_CORPUS, f, indent=2, ensure_ascii=False)
print("SPF corpus saved   → rag/spf_corpus.json")
print("FAISS index already saved → rag/spf_faiss.index")

print("\n" + "-" * 64)
print("  Agent 1 : Detection      (XLM-RoBERTa + language detection)")
print("  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)")
print("  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)")

SPF taxonomy saved → models/spf_taxonomy.json
SPF corpus saved   → rag/spf_corpus.json
FAISS index already saved → rag/spf_faiss.index

----------------------------------------------------------------
  Agent 1 : Detection      (XLM-RoBERTa + language detection)
  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)
  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)
